In [0]:
# Databricks notebook source
import sys
import os
from pyspark.sql.functions import lit

# --- Parameter Retrieval (Environment Layer) ---
dbutils.widgets.text("catalog_name", "")
dbutils.widgets.text("raw_path", "") 
dbutils.widgets.text("job_run_id", "") 

CATALOG = dbutils.widgets.get("catalog_name") 
RAW_PATH = dbutils.widgets.get("raw_path")
JOB_RUN_ID = dbutils.widgets.get("job_run_id")

if not CATALOG or not RAW_PATH:
    raise ValueError("CATALOG or RAW_PATH parameter is strictly required by the parent job.")

# --- Dynamic Root Discovery (Pathing Layer) ---
project_root = os.getcwd()
while project_root != '/' and not os.path.exists(os.path.join(project_root, 'src')):
    project_root = os.path.dirname(project_root)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# --- Explicit Library Imports (Modularity Layer) ---
from src.config.paths import ProjectConfig
from src.config.schemas import LOAN_PURPOSES_SCHEMA
from src.utils.spark_io import read_reference_source
from src.utils.audit import get_batch_id, add_ref_metadata

# Instantiate Configuration
cfg = ProjectConfig(catalog=CATALOG, raw_path=RAW_PATH)

In [0]:
# --- Pipeline Execution (Logic Layer) ---
batch_id = get_batch_id(JOB_RUN_ID)
print(f"Executing Customer Ingestion \nCatalog: {cfg.CATALOG} \nBatch ID: {batch_id}")


# Execute the read logic
df_lkp_raw = read_reference_source(
    spark=spark, 
    source_path=cfg.LOAN_PURPOSE_PATH, 
    schema=LOAN_PURPOSES_SCHEMA
).withColumn("_batch_id", lit(batch_id))

# 2. Enrich with Silver Metadata
df_lkp_enriched = add_ref_metadata(df_lkp_raw)

# Create Temp View
df_lkp_enriched.createOrReplaceTempView("tmp_loan_purpose_seed")

# Execute MERGE
spark.sql(
    f"""
    MERGE INTO {cfg.REF_LOAN_PURPOSES} AS target
    USING tmp_loan_purpose_seed AS source
    ON target.loan_purpose = source.loan_purpose
    
    WHEN MATCHED AND (target.is_active <> source.is_active) THEN
      UPDATE SET 
        target.is_active = source.is_active,
        target._updated_at = source._updated_at,
        target._source_file = source._source_file,
        target._processed_by = source._processed_by,
        target._batch_id = source._batch_id
        
    WHEN NOT MATCHED THEN
      INSERT (
        loan_purpose, 
        is_active,         
        _updated_at,
        _source_file, 
        _processed_by,
        _batch_id
      ) 
      VALUES (
        source.loan_purpose, 
        source.is_active, 
        source._updated_at,       
        source._source_file, 
        source._processed_by,
        source._batch_id
      )
"""
)

In [0]:
# %sql
# select * from lending_risk_dev.silver.ref_loan_purposes limit 10;